In [1]:
!pip install -q openai

In [2]:
from openai import OpenAI
from google.colab import userdata

client = OpenAI(
    api_key=userdata.get('GROQ_API_KEY'),
    base_url="https://api.groq.com/openai/v1"
)
MODEL = "openai/gpt-oss-20b"
print("API client setup completed!")

API client setup completed!


In [3]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "Say hello in one sentence."}
    ]
)
print(response.choices[0].message.content)

Hello!


In [5]:
messages = [
    {
        "role": "system",
        "content": "You are a friendly assistant. Give short answers."
    },
    {
        "role": "user",
        "content": "Hi, my name is Sruthi."
    },
    {
        "role": "assistant",
        "content": "Hello Sruthi! Nice to meet you."
    },
    {
        "role": "user",
        "content": "What is my name?"
    }
]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages
)

print(response.choices[0].message.content)

Your name is Sruthi.


In [6]:
SYSTEM = """
ROLE:
You are a friendly assistant for Sweet Crumbs Bakery.

FACTS:
We sell:
- Plum Cake: Rs. 500
- Chocolate Cake: Rs. 650
- Cupcake: Rs. 80

# RULES:
# - Answer politely.
# - Only provide information about our bakery.
# - Do not make up prices.

FORMAT:
Keep answers short and simple.
"""

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": "What cakes do you sell?"}
    ]
)

print(response.choices[0].message.content)

We sell:

- Plum Cake – Rs. 500  
- Chocolate Cake – Rs. 650  
- Cupcake – Rs. 80


In [7]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": "Can you deliver a cake in 1 hour?"}
    ]
)

print(response.choices[0].message.content)

I’m sorry, we don’t offer delivery.  
We sell:  
- Plum Cake – Rs. 500  
- Chocolate Cake – Rs. 650  
- Cupcake – Rs. 80


In [9]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": """
            Classify each customer message into one of these categories:
            ORDER, ENQUIRY, or COMPLAINT.

            Examples:
            Message: I want to buy two cupcakes.
            Category: ORDER

            Message: What is the price of chocolate cake?
            Category: ENQUIRY

            Message: My cake arrived damaged.
            Category: COMPLAINT

            Return only the category.
            """
        },
        {
            "role": "user",
            "content": "Can I order one plum cake?"
        }
    ]
)

print(response.choices[0].message.content)

ORDER


In [10]:
import json

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": """
            Extract the order details from the customer's message.
            Return only valid JSON with these fields:
            name, item, qty, eggless.
            """
        },
        {
            "role": "user",
            "content": "My name is Ravi. I want 2 plum cakes, both eggless."
        }
    ]
)

result = response.choices[0].message.content

print(result)

{"name":"Ravi","item":"plum cakes","qty":2,"eggless":true}


In [11]:
import json

try:
    order = json.loads(result)

    print("Customer name:", order["name"])
    print("Item:", order["item"])
    print("Quantity:", order["qty"])
    print("Eggless:", order["eggless"])

except json.JSONDecodeError:
    print("Invalid JSON response!")

Customer name: Ravi
Item: plum cakes
Quantity: 2
Eggless: True


In [13]:
MENU = {
    "plum cakes": 500,
    "chocolate cake": 650,
    "cupcake": 80
}

total = MENU[order["item"]] * order["qty"]

print("Item:", order["item"])
print("Quantity:", order["qty"])
print("Total price: Rs.", total)

Item: plum cakes
Quantity: 2
Total price: Rs. 1000


In [15]:
history = [
    {"role": "system", "content": "You are a friendly assistant."}
]

history.append({
    "role": "user",
    "content": "My name is Sruthi."
})

response = client.chat.completions.create(
    model=MODEL,
    messages=history
)

reply = response.choices[0].message.content

history.append({
    "role": "assistant",
    "content": reply
})

print(reply)

Nice to meet you, Sruthi! How can I help you today?


In [16]:
history.append({
    "role": "user",
    "content": "What is my name?"
})

response = client.chat.completions.create(
    model=MODEL,
    messages=history
)

reply = response.choices[0].message.content

history.append({
    "role": "assistant",
    "content": reply
})

print(reply)

You’re Sruthi.


In [17]:
MAX_TURNS = 2

recent_history = [history[0]] + history[1:][-MAX_TURNS:]

print(recent_history)

[{'role': 'system', 'content': 'You are a friendly assistant.'}, {'role': 'user', 'content': 'What is my name?'}, {'role': 'assistant', 'content': 'You’re Sruthi.'}]


In [18]:
response = client.chat.completions.create(
    model=MODEL,
    messages=recent_history + [
        {"role": "user", "content": "What is my name?"}
    ]
)

print(response.choices[0].message.content)

Your name is Sruthi.


In [19]:
MENU = {
    "plum cake": 500,
    "chocolate cake": 650,
    "cupcake": 80
}

def get_price(item: str, qty: int = 1) -> int:
    """Return the total price for the given bakery item and quantity."""
    return MENU[item.lower().strip()] * qty

print(get_price("plum cake", 2))
print(get_price("cupcake", 3))

1000
240


In [21]:
def my_decorator(func):
    def wrapper():
        print("Before function execution")
        func()
        print("After function execution")
    return wrapper

@my_decorator
def greet():
    print("Hello, Sruthi!")

greet()

Before function execution
Hello, Sruthi!
After function execution


In [22]:
from openai import RateLimitError, APIConnectionError, AuthenticationError

try:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": "Hello!"}
        ]
    )

    print(response.choices[0].message.content)

except RateLimitError:
    print("Too many requests. Please try again later.")

except AuthenticationError:
    print("Invalid API key. Please check your key.")

except APIConnectionError:
    print("Connection problem. Please check your internet.")

except Exception as e:
    print("Something went wrong:", e)

Hi there! 👋 How can I help you today?


In [23]:
MENU = {
    "plum cake": 500,
    "chocolate cake": 650,
    "cupcake": 80
}

SYSTEM = """
You are a friendly customer support assistant for Sweet Crumbs Bakery.

MENU:
- Plum Cake: Rs. 500
- Chocolate Cake: Rs. 650
- Cupcake: Rs. 80

RULES:
- Answer politely.
- Only provide bakery-related information.
- Never invent prices.
- If you don't know something, say so.
- Keep answers short and simple.
"""

history = [
    {"role": "system", "content": SYSTEM}
]

MAX_TURNS = 8

print("Sweet Crumbs AI Chatbot is ready!")

Sweet Crumbs AI Chatbot is ready!


In [24]:
from openai import RateLimitError, APIConnectionError, AuthenticationError

def chat(user_text: str) -> str:
    """Send a customer message to the bakery AI and return its reply."""

    history.append({
        "role": "user",
        "content": user_text
    })

    recent = [history[0]] + history[1:][-MAX_TURNS:]

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=recent,
            temperature=0.3
        )

        reply = response.choices[0].message.content

    except RateLimitError:
        reply = "We're a little busy. Please try again in a minute."

    except AuthenticationError:
        reply = "Authentication failed. Please check the API key."

    except APIConnectionError:
        reply = "I can't reach the server right now. Please try again."

    except Exception as e:
        reply = f"Something went wrong: {e}"

    history.append({
        "role": "assistant",
        "content": reply
    })

    return reply

print("Chat function created successfully!")

Chat function created successfully!


In [25]:
print(chat("Hi! What items are available in your bakery?"))

We have:

- Plum Cake – Rs. 500  
- Chocolate Cake – Rs. 650  
- Cupcake – Rs. 80  

Let us know if you’d like to order or need more details!


In [26]:
print(chat("What is the price of the chocolate cake?"))

Chocolate Cake – Rs. 650.


In [27]:
print(chat("Which bakery item did I ask about just now?"))

You asked about the Chocolate Cake.


In [28]:
print(chat("My favourite item is cupcake."))
print(chat("What is your bakery timing?"))
print(chat("Do you offer chocolate cake?"))

Nice choice! A cupcake is Rs. 80.
I’m sorry, but I don’t have the bakery’s operating hours.
Yes, we offer Chocolate Cake for Rs. 650.


In [29]:
print(chat("What is my favourite item?"))

Your favourite item is a cupcake.
